# NB05b — Decision Trees, Ensembles and XGBoost for Tabular Finance

**Role:** Core  
**Manual section:** 2.3.1.b — Tree-based models and ensembles  
**Prerequisites:** NB05

---

## Purpose of this notebook

This notebook compares tree-based methods against the logistic regression baseline from NB05. You will see how decision trees, random forests, gradient boosting and XGBoost perform on the same attrition dataset, and learn to evaluate whether the improvement justifies the added complexity.

**Dataset:** `dataset_C_attrition_model_ready.csv`

> **Core route notebook**  
> This notebook continues the same running attrition (also called churn) project as **NB04 → NB04b → NB05**.

> **Shared project file**  
> We load the same canonical modelling-table dataset used in NB05:  
> `dataset_C_attrition_model_ready.csv`

> **Optional short extensions (10–15 min each)**  
> - **NB05c** for Naïve Bayes, SVM and k-NN  
> - **NB05d** for a very small PyTorch neural-network example

---

## Section 0 — Why Move Beyond Linear Baselines

In Notebook 05 we fitted a **logistic regression** baseline. It achieved:
- AUC-ROC ≈ 0.87 (good ranking of attrition cases vs. retained customers),
- but recall ≈ 0.45 at the default 0.5 threshold — fewer than half of the actual attrition
  cases were caught.

**Why does this happen?**

Logistic regression draws a single straight boundary in feature space. Some attrition patterns
may be more complex:

- **Non-linearity:** attrition risk may jump sharply when `complaints_last_6m` exceeds a
  threshold, rather than increasing linearly.
- **Interactions:** high age *combined with* low activity may matter more than either
  feature alone.
- **Threshold-like behaviour:** customers with exactly 1 product may behave very
  differently from those with 2 or 3.

A model that can learn these patterns might improve recall significantly.

> **Goal of this notebook:** compare logistic regression against tree-based methods
> (decision tree, random forest, XGBoost) on the same bank-attrition data and the same
> evaluation framework.

---

## Data Setup — Load the Shared Project Table

We deliberately load the **same canonical modelling-table file** created in NB04 and validated in NB04b.

This is how a real project should feel:

- one business problem,
- one modelling table,
- several competing models.

That continuity is what makes the model comparison meaningful.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

_candidates = [Path('data/processed'), Path('../data/processed')]
DATA_DIR = next((p for p in _candidates if p.is_dir()), None)
if DATA_DIR is None:
    raise FileNotFoundError(
        'Cannot locate data/processed/. Launch the notebook from the project root or notebooks/.'
    )

MODEL_PATH = DATA_DIR / 'dataset_C_attrition_model_ready.csv'
if not MODEL_PATH.exists():
    raise FileNotFoundError(
        'Shared project dataset not found. Run NB04 and NB04b first so that '
        '`dataset_C_attrition_model_ready.csv` exists.'
    )

model_ready = pd.read_csv(MODEL_PATH)

target_col = 'attrition_flag'
id_col = 'customer_id'

y = model_ready[target_col]
X = model_ready.drop(columns=[target_col, id_col])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_sc = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_sc  = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

print(f'Loaded canonical project table: {MODEL_PATH.name}')
print(f'Train shape: {X_train.shape}')
print(f'Test shape:  {X_test.shape}')

### Reproduce the logistic regression baseline

We re-fit the same logistic regression from Notebook 05 so all models are evaluated on the
identical train/test split.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score)

logreg = LogisticRegression(max_iter=1000, random_state=42)
logreg.fit(X_train_sc, y_train)

y_pred_lr = logreg.predict(X_test_sc)
y_prob_lr = logreg.predict_proba(X_test_sc)[:, 1]

def eval_model(y_true, y_pred, y_prob, name="Model"):
    """Return a dict of standard classification metrics."""
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "F1": f1_score(y_true, y_pred),
        "AUC-ROC": roc_auc_score(y_true, y_prob),
    }

results = []
results.append(eval_model(y_test, y_pred_lr, y_prob_lr, "Logistic Regression"))
print("Logistic Regression baseline reproduced.")
for k, v in results[0].items():
    if k != "Model":
        print(f"  {k:<12s} {v:.3f}")

---

## Section 1 — Decision-Tree Intuition

### How does a decision tree work?

Imagine you are a bank manager trying to decide which customers are at risk of leaving.
You might ask a series of yes/no questions:

```
Is the customer active?
├── Yes → Are they older than 50?
│         ├── Yes → moderate risk
│         └── No  → low risk
└── No  → Do they have complaints?
          ├── Yes → HIGH risk
          └── No  → moderate risk
```

A **decision tree** learns these rules automatically from the data. At each step
(called a **split**), it picks:
- the **feature** that separates attrition cases from retained customers most effectively,
- the **threshold** value for that feature.

The process repeats until a stopping criterion is met (e.g., maximum depth, minimum
samples in a leaf).

### Strengths
- Very **interpretable**: you can read the rules like a flowchart.
- Naturally handles **non-linearity** and **interactions**.
- **No scaling needed**: the splits are based on rank order, not magnitude.

### Weakness
- A single tree tends to **overfit**: it memorises the training data and performs
  poorly on new data, especially if it grows deep.

---

## Section 2 — First Shallow Decision Tree

We fit a **shallow** tree (`max_depth=4`) to keep it interpretable and reduce overfitting.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

# Fit a shallow decision tree (unscaled features — trees don't need scaling)
dt = DecisionTreeClassifier(max_depth=4, random_state=42)
dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)
y_prob_dt = dt.predict_proba(X_test)[:, 1]

results.append(eval_model(y_test, y_pred_dt, y_prob_dt, "Decision Tree (depth=4)"))

print("Decision Tree (depth=4) — Test Metrics:")
for k, v in results[-1].items():
    if k != "Model":
        print(f"  {k:<12s} {v:.3f}")

In [ ]:
# Visualise the tree
fig, ax = plt.subplots(figsize=(20, 8))
plot_tree(dt, feature_names=list(X_train.columns),
          class_names=["Stayed", "Churned"],
          filled=True, rounded=True, fontsize=8, ax=ax,
          impurity=False, proportion=True)
ax.set_title("Shallow Decision Tree (max_depth=4)", fontsize=14)
plt.tight_layout()
plt.show()

### Reading the tree

- Each **box** shows the splitting rule (e.g., `age <= 42.5`), the proportion of classes,
  and the predicted label.
- **Orange** boxes lean towards *attrited*; **blue** boxes lean towards *stayed*.
- Deeper boxes capture more specific sub-groups.

> **Reflection:** look at the first split. Does the feature and threshold make business
> sense for customer attrition?

### Overfitting check — what if we let the tree grow unconstrained?

In [ ]:
# Deep tree — no depth limit
dt_deep = DecisionTreeClassifier(random_state=42)
dt_deep.fit(X_train, y_train)

train_acc = dt_deep.score(X_train, y_train)
test_acc  = dt_deep.score(X_test, y_test)

print(f"Unrestricted tree — Training accuracy: {train_acc:.3f}")
print(f"Unrestricted tree — Test accuracy:     {test_acc:.3f}")
print(f"Gap: {train_acc - test_acc:.3f}")
print()
print("The large gap shows the tree has memorised the training data.")
print("This is classic OVERFITTING.")

**Key insight:** a single deep tree achieves near-perfect training accuracy but much
lower test accuracy. It has learned noise, not genuine patterns.

The shallow tree (depth 4) trades some training accuracy for much better generalisation.

> *"A model that memorises is not a model that understands."*

---

## Section 3 — From One Tree to Ensembles (Random Forest)

### The problem with a single tree

Even a carefully pruned tree can be **unstable**: small changes in the training data
can produce a very different set of rules.

### The ensemble idea

What if we trained **many trees**, each on a slightly different random subset of the
data, and then **averaged** their predictions?

- Each individual tree may be imperfect, but their errors tend to cancel out.
- The result is more **stable** and usually more accurate.

This is the core idea behind **Random Forest**:
1. Draw many random bootstrap samples from the training data.
2. Fit one decision tree per sample, with a random subset of features at each split.
3. Combine predictions by majority vote (classification) or averaging (regression).

> *"The wisdom of the crowd often beats the smartest individual."*

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42,
                             n_jobs=-1)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

results.append(eval_model(y_test, y_pred_rf, y_prob_rf, "Random Forest"))

print("Random Forest — Test Metrics:")
for k, v in results[-1].items():
    if k != "Model":
        print(f"  {k:<12s} {v:.3f}")

The random forest typically improves over a single tree on both AUC and stability.

> **Question:** compare the recall of the random forest to the logistic regression
> baseline. Did the non-linear model catch more attrition cases?

---

## Section 4 — Gradient-Boosting Intuition

**Random Forest** trains many trees *independently* and averages them.

**Gradient boosting** takes a different approach: it trains trees *sequentially*, where
each new tree focuses on the **mistakes of the previous one**.

### Analogy

Imagine a student taking a quiz:
1. **Attempt 1:** the student answers all questions — gets some right, some wrong.
2. **Attempt 2:** the student studies only the questions they got wrong and tries again.
3. **Attempt 3:** they focus on the *remaining* mistakes.
4. After several rounds, the combined answers are much better than any single attempt.

This is exactly what gradient boosting does with decision trees:
- Tree 1 makes predictions.
- Tree 2 is trained on the *residual errors* of Tree 1.
- Tree 3 on the residual errors of Trees 1 + 2.
- And so on.

### Why is this powerful for tabular data?

- Boosting naturally focuses on the **hard examples** — the customers that are difficult
  to classify.
- It can learn **subtle patterns** that a single model or a simple average might miss.
- On structured/tabular data (spreadsheets, databases), boosted trees are often the
  **top-performing** method — frequently winning data-science competitions.

> **XGBoost** (eXtreme Gradient Boosting) is the most widely-used implementation of this
> idea.

---

## Section 5 — XGBoost: The Main Practical AI Method for Tabular Finance

**XGBoost** is practically the industry standard for structured/tabular data in finance.

Why practitioners choose it:
- **Strong performance** out of the box.
- **Handles missing values** internally (though our dataset has none).
- **Built-in regularisation** to reduce overfitting.
- **Speed** — optimised C++ core with parallel computation.
- Battle-tested in **credit scoring, attrition prediction, fraud detection, and insurance
  pricing**.

Let's fit a first XGBoost classifier with sensible defaults.

In [ ]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss",
)
xgb.fit(X_train, y_train)

y_pred_xgb = xgb.predict(X_test)
y_prob_xgb = xgb.predict_proba(X_test)[:, 1]

results.append(eval_model(y_test, y_pred_xgb, y_prob_xgb, "XGBoost"))

print("XGBoost — Test Metrics:")
for k, v in results[-1].items():
    if k != "Model":
        print(f"  {k:<12s} {v:.3f}")

---

## Section 6 — A Small Hyperparameter Experiment

XGBoost has many tuning knobs. We focus on three that matter most in practice:

| Parameter | What it controls | Typical range |
|-----------|-----------------|--------------|
| `n_estimators` | Number of boosting rounds (trees) | 100 – 1 000 |
| `max_depth` | Maximum depth of each tree | 3 – 8 |
| `learning_rate` | Step size per round (lower = more conservative) | 0.01 – 0.3 |

### Experiment: compare four configurations

We try a few hand-picked settings to build intuition — not an exhaustive search.

In [ ]:
configs = [
    {"n_estimators": 100, "max_depth": 3, "learning_rate": 0.10, "label": "A: shallow, fast"},
    {"n_estimators": 200, "max_depth": 5, "learning_rate": 0.10, "label": "B: default"},
    {"n_estimators": 400, "max_depth": 5, "learning_rate": 0.05, "label": "C: more trees, slower lr"},
    {"n_estimators": 300, "max_depth": 7, "learning_rate": 0.05, "label": "D: deeper trees"},
]

hp_results = []
for cfg in configs:
    m = XGBClassifier(
        n_estimators=cfg["n_estimators"],
        max_depth=cfg["max_depth"],
        learning_rate=cfg["learning_rate"],
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric="logloss",
    )
    m.fit(X_train, y_train)
    yp = m.predict(X_test)
    ypr = m.predict_proba(X_test)[:, 1]
    hp_results.append({
        "Config": cfg["label"],
        "n_est": cfg["n_estimators"],
        "depth": cfg["max_depth"],
        "lr": cfg["learning_rate"],
        "Accuracy": accuracy_score(y_test, yp),
        "Recall": recall_score(y_test, yp),
        "F1": f1_score(y_test, yp),
        "AUC-ROC": roc_auc_score(y_test, ypr),
    })

hp_df = pd.DataFrame(hp_results)
hp_df[["Config", "n_est", "depth", "lr", "Accuracy", "Recall", "F1", "AUC-ROC"]].round(3)

### Interpretation

- Increasing `n_estimators` with a lower `learning_rate` often improves AUC but adds
  training time.
- Deeper trees (`max_depth=7`) may overfit — check if the gain is real.
- The differences between configurations are often **small**, which is typical of
  well-behaved tabular datasets.

> **Practical rule of thumb:** a moderately-tuned XGBoost is usually sufficient. Spending
> hours on hyperparameter search rarely changes the business decision.

---

## Section 7 — Side-by-Side Model Comparison

Let's bring all four models together in a single table.

In [ ]:
comparison = pd.DataFrame(results)
comparison = comparison.set_index("Model").round(3)
comparison

In [ ]:
# Visual comparison — AUC and Recall
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

models = comparison.index.tolist()
x = range(len(models))

axes[0].barh(models, comparison["AUC-ROC"], color="#5b9bd5", edgecolor="white")
axes[0].set_xlabel("AUC-ROC")
axes[0].set_title("AUC-ROC by Model")
axes[0].set_xlim(0.5, 1.0)

axes[1].barh(models, comparison["Recall"], color="#ed7d31", edgecolor="white")
axes[1].set_xlabel("Recall")
axes[1].set_title("Recall by Model")
axes[1].set_xlim(0.0, 1.0)

plt.tight_layout()
plt.show()

### What do we gain from extra complexity?

The baseline-first logic of the course says that a stronger model must justify its added complexity.

So after fitting all models, we compare them **directly against logistic regression**.


In [ ]:
baseline = comparison.loc['Logistic Regression']

gain_vs_baseline = comparison.subtract(baseline).drop(index='Logistic Regression')
gain_view = gain_vs_baseline[['Accuracy', 'Recall', 'AUC-ROC']].round(3)
print('Gain vs logistic-regression baseline:')
display(gain_view)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
gain_view['AUC-ROC'].plot(kind='bar', ax=axes[0], title='Δ AUC vs Logistic Regression', color='#5b9bd5')
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].set_ylabel('Change in AUC')

gain_view['Recall'].plot(kind='bar', ax=axes[1], title='Δ Recall vs Logistic Regression', color='#ed7d31')
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_ylabel('Change in Recall')

plt.tight_layout()
plt.show()

### What does this tell us?

| Observation | Implication |
|-------------|------------|
| XGBoost and Random Forest typically have higher AUC than logistic regression | Non-linear patterns in the data are real |
| Recall improvements vary | The default 0.5 threshold may still not be optimal — threshold tuning matters for all models |
| The shallow decision tree is competitive but less stable | Overfitting risk is real without ensembling |
| Logistic regression is still a respectable baseline | Simple models should not be dismissed too quickly |

> **Key question:** *"Is the improvement from XGBoost large enough to justify the extra
> complexity?"* — There is no universal answer. It depends on the business context.

---

## Section 8 — Feature Importance and Interpretation

XGBoost can tell us which features it relied on most. This is useful for:
- **Understanding** the model's logic.
- **Communicating** results to stakeholders.
- **Validating** that the model uses sensible signals (not noise or artefacts).

> **Caution:** feature importance shows **what the model uses**, not necessarily **what
> causes** attrition. Correlation ≠ causation.

In [ ]:
# XGBoost feature importance (gain-based)
fi = pd.DataFrame({
    "feature": X_train.columns,
    "importance": xgb.feature_importances_,
}).sort_values("importance", ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(fi["feature"], fi["importance"], color="#70ad47", edgecolor="white")
ax.set_xlabel("Feature importance (gain)")
ax.set_title("XGBoost — Feature Importance")
plt.tight_layout()
plt.show()

### Reading the chart

- The most important features are those that contribute most to reducing prediction
  error across all trees.
- Compare with the logistic regression coefficients from Notebook 05 — do the same
  features appear as influential?

> **Reflection:** does the ranking make business sense? For example, if `age` or
> `complaints_last_6m` rank high, that aligns with banking intuition about attrition
> drivers.

---

## Section 9 — Business Recommendation and Model Limits

### Writing a stakeholder recommendation

Imagine you are presenting to the bank's Head of Customer Retention. Based on the model
comparison, write a **short recommendation** (3–5 sentences) covering:

1. Which model do you recommend for a first deployment, and why?
2. What is the expected performance level (cite a key metric)?
3. What are the main risks or limitations?

### Model limitations to keep in mind

| Limitation | Why it matters |
|-----------|---------------|
| **One-time split evaluation** | We tested on a single 80/20 split. Cross-validation would give more robust estimates. |
| **No temporal validation** | In reality, we should test on *future* data, not a random split. |
| **Class imbalance** | ~20 % attrition. The model may still miss many attrition cases. |
| **Feature scope** | We use only the features in Dataset C. Additional data (transaction history, call-centre logs) might improve predictions. |
| **No causal claims** | The model identifies *associations*, not causes. An intervention based on model scores needs careful design. |

> *"A good model is the beginning of a conversation with the business, not the end."*

---

## Section 10 — Closing Bridge to Validation and Production Thinking

We now have a strong baseline comparison: logistic regression → decision tree → random
forest → XGBoost. But a single train/test split is only the first step towards
trustworthy deployment.

In **Notebook 06** we will address:

| Topic | Why it matters |
|-------|---------------|
| **Cross-validation** | Evaluate model stability across multiple folds |
| **Regularisation** | Control overfitting more systematically |
| **Class imbalance handling** | Strategies for the ~80/20 stayed/attrited split |
| **Threshold tuning** | Choosing a cut-off that balances precision and recall |
| **Business interpretation** | Translating model outputs into actionable insights |

> *"The best model is useless if its evaluation is unreliable."*

### Optional route reminder

If you want to see additional supervised methods on similar data:

- open **NB05c** for Naïve Bayes, SVM and k-NN;
- open **NB05d** for a compact PyTorch neural-network example.

For the main route, continue to **NB06** for threshold choice, cross-validation and business interpretation.


---

### Self-practice tasks

1. **Hyperparameter exploration:** change one XGBoost parameter (e.g., set
   `max_depth=3`) and observe how accuracy, recall and AUC change. Explain the result
   in one sentence.
2. **Interpretability comparison:** write 2–3 sentences comparing the interpretability of
   logistic regression and XGBoost.
3. **Three reasons:** list three reasons why XGBoost often performs well on tabular
   business data.
4. **Validation next step:** propose one additional validation step you would take before
   deploying this model in a real bank.

---

### Recap — what we learned

| Concept | Key takeaway |
|---------|-------------|
| Decision trees | Learn if/then rules; interpretable but prone to overfitting |
| Ensembles (Random Forest) | Average many trees → more stable predictions |
| Gradient Boosting (XGBoost) | Sequential correction of errors → often best on tabular data |
| Model comparison | Always compare against a simple baseline |
| Feature importance | Useful for interpretation, but ≠ causation |

---

*End of Notebook 05b — Decision Trees, Ensembles and XGBoost for Tabular Finance*

---

**Project chain:** NB04 (build table) → NB04b (treat data) → NB05 (baseline) → **NB05b (trees)** → NB06 (validate & interpret)  
**Current position:** NB05b